# Image Stitching Template v1.0.0
___
Jonathan Zhang

Here, we outline how to use this notebook to stitch and background subtract images from HT-BAM experiments. Before you start, carefully read the guidelines below.

## Stitching

### Metadata framework
There exist two sources of metadata to bridge the gap between raw pixels and experimental variables:
- `imaging.csv`: This file logs hardware state and raster metadata for all images taken in an experiment. All information needed to stitch and background subtract images is found here. You may manually overwrite this file to alter how images are processed, but always be sure to create a backup copy beforehand.
    - For the sake of continuity, the stitching and background subtraction modules carry over this data into `stitched_images.csv` and `bgsub_images.csv`.
- `series_index.json`: Metadata specific to the experimental conditions (e.g., titration concentrations) is saved within individual series directories. This ensures that raw data remain linked to their biochemical context. This file is largely irrelevant during stitching, but become important at the processing stage.

### Image stitching Logic
The stitching process is designed to be automated and data-driven:
- `stitch_images`: Once the root path is provided, this function acts as the primary orchestrator. It uses the raster parameters defined in `imaging.csv` to find and stitch all raw data associated with the experiment.
- `stitch_single_raster`: This is the lower-level function called by the main stitcher. If your file-naming convention or folder structure changes substantially from convention, you can bypass the automated discovery logic implemented in `stitch_images` and use this function directly by providing your own path-finding logic.

### Important considerations
Some important considerations before beginning:
- Manually delete any images you do not want processed beforehand.
- No manual reorginization. Previous versions of the stitching code required manually directory reorginization. In this version, reorganizing the directory structure is entirely unnecessary and will likely cause the code to fail.
- Do not rename, move, or create new files within directories of raw images or outputs from the stitching and background subtraction modules (i.e. `raw_images`, `stitched_images`, and `bgsub_images`).

## Background Subtraction

### Background subtraction logic
Like stitching, background subtraction logic is split into two layers to balance automation with developer flexibility:
- `background_subtract_images`: The main function for background subtraction. It automatically groups images based on specified metadata headers (like channel or exposure_time) and applies the subtraction across the entire experiment. It uses `stitched_images.csv` as the source for metadata.
- `background_subtract`: The lower-level function, called within the above, which performs the background subtraction operation on a single image or array. If your file structure changes substantially or you need to implement custom subtraction logic, you can call this function directly and bypass the automated grouping.

### Image grouping
Users can specify metadata headers in `stitched_images.csv` to use for image grouping (i.e. exposure time, lightsource, etc.). If complex groupings are required, one can manually write new columns to `stitched_images.csv` to define custom logic (but always save a backup copy beforehand).

In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
from htbam_analysis.stitching.stitching import stitch_images, background_subtract_images
from pathlib import Path
import pandas as pd
import jax

import pandas as pd
from pathlib import Path

jax.default_device = jax.devices('cpu')[0]

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


### Stitch images

The `stitch_images` function requires three arguments:
1. `root`: path to the folder containing experiment data.
2. `rotation`: global rotation parameter. This can drift over time and may need to be adjusted. If stitched images look jagged, this is the first thing that should be tuned.
3. `acqui_ori`: specifies corner of the device where imaging starts. In our experiments, this is always from the top-right (i.e. `(True, False)`) 

In [10]:
# specify parameters for stitching
root = Path('/Volumes/JSZ2/deez')

# stitch images
stitch_images(
    root = root,
    rotation = -2,
    acqui_ori=(True, False)
)

Stitching image rasters:   3%|▎         | 2/76 [00:14<09:03,  7.35s/it]

Error stitching /Volumes/JSZ2/deez/raw_images/2026-03-05_12-13-17_standard_curve_NADH/2026-03-05_12-13-32_d1_aura_UV340_5_1000_dynamic_range_2x2_4_standard_curve_NADH_0: [Errno 2] No such file or directory: '/Volumes/JSZ2/deez/raw_images/2026-03-05_12-13-17_standard_curve_NADH/2026-03-05_12-13-32_d1_aura_UV340_5_1000_dynamic_range_2x2_4_standard_curve_NADH_0/Pos-1-000_000.tif'
Error stitching /Volumes/JSZ2/deez/raw_images/2026-03-05_12-13-17_standard_curve_NADH/2026-03-05_12-13-32_d1_aura_UV340_5_250_dynamic_range_2x2_4_standard_curve_NADH_0: [Errno 2] No such file or directory: '/Volumes/JSZ2/deez/raw_images/2026-03-05_12-13-17_standard_curve_NADH/2026-03-05_12-13-32_d1_aura_UV340_5_250_dynamic_range_2x2_4_standard_curve_NADH_0/Pos-1-000_000.tif'
Error stitching /Volumes/JSZ2/deez/raw_images/2026-03-05_12-13-17_standard_curve_NADH/2026-03-05_12-13-32_d1_aura_UV340_5_500_dynamic_range_2x2_4_standard_curve_NADH_0: [Errno 2] No such file or directory: '/Volumes/JSZ2/deez/raw_images/2026-

Stitching image rasters: 100%|██████████| 76/76 [06:55<00:00,  5.46s/it]

62 / 76 rasters were successfully stitched


### Background subtract images



In [ ]:
# a list of paths to background images
background_images = [] 

# specify settings to group images by
# the settings below are standard
settings_to_match = ['temp', 'hum','setup', 'dname', 'lightsource', 'channel', 'exposure', 'camera_mode', 'binning', 'nosepiece']

# match images by metadata fields and perform background subtraction
background_subtract_images(
    root = root,
    background_images = background_images,
    settings_to_match = settings_to_match
)

In [ ]:
# #region IO and sanity checks

# # check that root is a Path object and exists
# if not isinstance(root, Path):
#     root = Path(root)
# assert root.exists(), f"Root directory {root} does not exist"

# # check that background images are Path objects and exist
# for i in range(len(background_images)):
#     if not isinstance(background_images[i], Path):
#         background_images[i] = Path(background_images[i])
#     assert background_images[i].exists(), f"Background image {background_images[i]} does not exist"

# # check that stitched images directory and csv exist
# stitched_images = root / 'stitched_images'
# assert stitched_images.exists(), f"Stitched images directory {stitched_images} does not exist"

# stitched_images_csv = stitched_images / 'stitched_images.csv'
# assert stitched_images_csv.exists(), f"Stitched images csv {stitched_images_csv} does not exist"

# # load stitched images dataframe
# stitched_image_df = pd.read_csv(stitched_images_csv)
# stitched_image_df['image_path'] = stitched_image_df['image_path'].apply(Path)
# stitched_image_df['image_path'] = stitched_image_df['image_path'].apply(lambda f: stitched_images / f)

# #endregion

# # group rows by imaging settings
# grouped = stitched_image_df.groupby(by=settings_to_match, dropna=False)

# bgsub_images = root / 'bgsub_images'
# bgsub_df = []
# for key, group in grouped:

#     # region background image sanity checks

#     print('Attempting background subtracting on images with the following settings:')
#     print(' | '.join(['{}: {}'.format(setting, value) for setting, value in zip(settings_to_match, key)]))

#     # check if a unique background image can be found
#     background_mask = group['image_path'].isin(background_images)

#     if sum(background_mask) == 0:
#         print('No background image found. Continuing.\n')
#         continue

#     elif sum(background_mask) > 1:
#         print('Too many background images found. Continuing.\n')    
#         continue

#     # endregion

#     # df for background image (should just be one row)
#     background_df = group.loc[background_mask].copy()
#     background_image_path = group[background_mask].iloc[0]['image_path']
#     background_image = io.imread(background_image_path)

#     # df for target images
#     target_df = group.loc[~background_mask].copy()
#     target_df['background_image'] = [background_image_path] * len(target_df)
#     target_images = target_df['image_path'].tolist()

#     success_mask = np.ones(len(target_df), dtype=bool)
#     for i in tqdm.tqdm(range(len(target_images)), desc='Running background subtraction.'):
#         target_image_path = target_images[i]
#         try: 
#             target_image = io.imread(target_image_path)
#             subtracted_image = backgroud_subtract(background_image, target_image)
            
#             # save the stitched image
#             outfile = bgsub_images / target_image_path.relative_to(stitched_images)
#             outfile.parent.mkdir(parents=True, exist_ok=True)
#             io.imsave(outfile, subtracted_image, plugin="tifffile", check_contrast=False)

#         except Exception as e:
#             print(f"Error processing {target_image_path}: {e}")
#             success_mask[i] = False
    
#     print("{successes} / {total} images were successfully background subtracted".format(successes=(success_mask).sum(), total=len(success_mask)))
#     print()

#     bgsub_df.append(target_df[success_mask])

# # format and save the background subtraction dataframe
# bgsub_df = pd.concat(bgsub_df, ignore_index=True)
# bgsub_df['image_path'] = bgsub_df['image_path'].apply(lambda f: f.relative_to(stitched_images))
# bgsub_df['background_image'] = bgsub_df['background_image'].apply(lambda f: f.relative_to(stitched_images))
# bgsub_df.to_csv(bgsub_images / 'bgsub_images.csv', index=False)

# # carry over metadata
# for src in stitched_images.rglob('*index.csv'):
#     dst = bgsub_images / src.relative_to(stitched_images)
#     copy(src, dst)

Attempting background subtracting on images with the following settings:
temp: nan | hum: nan | setup: Pinney_Setup_2 | dname: d1 | lightsource: aura_UV340 | channel: 5 | exposure: 500 | camera_mode: dynamic_range | binning: 2x2 | nosepiece: 4


Running background subtraction.: 100%|██████████| 114/114 [03:25<00:00,  1.80s/it]


114 / 114 rasters were successfully background subtracted

Attempting background subtracting on images with the following settings:
temp: nan | hum: nan | setup: Pinney_Setup_2 | dname: d1 | lightsource: aura_cyan | channel: 2 | exposure: 5 | camera_mode: sensitivity | binning: 2x2 | nosepiece: 1


Running background subtraction.: 100%|██████████| 2/2 [00:03<00:00,  1.51s/it]

2 / 2 rasters were successfully background subtracted

